In [1]:
import random
import matplotlib.pyplot as plt

def simulate_average_case(N, num_trials=1000):
    total_max_depth = 0
    
    for _ in range(num_trials):
        # Generujemy losowy szereg czasowy (przeciętny przypadek)
        # Używamy losowej permutacji liczb od 1 do N
        f_core = random.sample(range(1, N + 1), N)
        
        # Zgodnie z publikacją, rozszerzamy szereg o globalne minimum na początku 
        # i globalne maksimum na końcu: f(0) < f(k) < f(N+1)
        f = [float('-inf')] + f_core + [float('inf')]
        
        Top = []
        Bot = []
        Dir = 1
        
        max_stack_depth = 0
        
        # Główna pętla algorytmu
        for t in range(1, len(f)):
            # Algorytm z publikacji (Appendix A)
            if (f[t] - f[t-1]) * Dir < 0:
                if Dir == 1:
                    Top.append(f[t-1])
                else:
                    Bot.append(f[t-1])
                Dir = -Dir
            else:
                # Uwaga: Publikacja używa 'if', ale w praktycznych implementacjach 
                # (tzw. Elder Rule) stosuje się pętlę 'while', aby poprawnie zdjąć
                # wszystkie szczyty, które właśnie pokonaliśmy.
                while Top and Bot and ((Dir == 1 and f[t] > Top[-1]) or (Dir == -1 and f[t] < Bot[-1])):
                    Top.pop()
                    Bot.pop()
            
            # Śledzimy maksymalne zapotrzebowanie na pamięć (głębokość stosu)
            current_depth = len(Top) + len(Bot)
            if current_depth > max_stack_depth:
                max_stack_depth = current_depth
                
        total_max_depth += max_stack_depth
        
    return total_max_depth / num_trials

# --- Sprawdzamy w kodzie ---
N_values = [100, 500, 1000, 5000, 10000]
average_depths = []

print("Rozmiar danych (N) | Średnia maks. głębokość stosu")
print("-" * 50)

for N in N_values:
    avg_depth = simulate_average_case(N, num_trials=100)
    average_depths.append(avg_depth)
    print(f"{N:<18} | {avg_depth:.2f}")

Rozmiar danych (N) | Średnia maks. głębokość stosu
--------------------------------------------------
100                | 18.54
500                | 29.68
1000               | 34.50
5000               | 51.02
10000              | 55.72


In [2]:
import numpy as np

def simulate_brownian_with_drift(N, drift=0.0, dt=1.0, num_trials=1000):
    """
    Simulates the barcode algorithm stack depth for a Brownian motion with drift.
    
    Args:
        N: Number of time steps.
        drift: The constant drift parameter (m).
        dt: Time step size.
        num_trials: Number of independent trajectories to simulate.
        
    Returns:
        Tuple containing average max depths: (total, top_stack, bottom_stack)
    """
    total_max_depth = 0
    total_max_top = 0
    total_max_bot = 0
    
    for _ in range(num_trials):
        # Generate Gaussian steps for Brownian motion with drift
        # dx = drift * dt + sqrt(dt) * N(0, 1)
        stdev = np.sqrt(dt)
        mean_step = drift * dt
        steps = np.random.normal(loc=mean_step, scale=stdev, size=N)
        
        # Cumulative sum to create the trajectory
        f_core = np.cumsum(steps).tolist()
        
        # Augment with global extrema as per the paper's assumptions
        f = [float('-inf')] + f_core + [float('inf')]
        
        top_stack = []
        bot_stack = []
        direction = 1
        
        max_depth = 0
        max_top = 0
        max_bot = 0
        
        # Main algorithm loop
        for t in range(1, len(f)):
            if (f[t] - f[t-1]) * direction < 0:
                # Direction changed, push the extremum
                if direction == 1:
                    top_stack.append(f[t-1])
                else:
                    bot_stack.append(f[t-1])
                direction = -direction
            else:
                # Same direction, check if we hit a wall to pop pairs (Elder Rule)
                while top_stack and bot_stack and \
                      ((direction == 1 and f[t] > top_stack[-1]) or \
                       (direction == -1 and f[t] < bot_stack[-1])):
                    top_stack.pop()
                    bot_stack.pop()
            
            # Track memory usage independently
            current_depth = len(top_stack) + len(bot_stack)
            
            if current_depth > max_depth: max_depth = current_depth
            if len(top_stack) > max_top: max_top = len(top_stack)
            if len(bot_stack) > max_bot: max_bot = len(bot_stack)
            
        total_max_depth += max_depth
        total_max_top += max_top
        total_max_bot += max_bot
        
    return (total_max_depth / num_trials, 
            total_max_top / num_trials, 
            total_max_bot / num_trials)

# --- Run experiments ---
N = 5000
drifts = [0.0, 0.05, 0.1, 0.5] # 0.0 is pure Brownian motion

print(f"{'Drift':<10} | {'Total Depth':<15} | {'Top Stack (Max)':<20} | {'Bot Stack (Min)':<20}")
print("-" * 75)

for m in drifts:
    avg_total, avg_top, avg_bot = simulate_brownian_with_drift(N, drift=m, num_trials=200)
    print(f"{m:<10.2f} | {avg_total:<15.2f} | {avg_top:<20.2f} | {avg_bot:<20.2f}")

Drift      | Total Depth     | Top Stack (Max)      | Bot Stack (Min)     
---------------------------------------------------------------------------
0.00       | 18.61           | 9.57                 | 9.03                
0.05       | 17.45           | 8.98                 | 8.47                
0.10       | 16.58           | 8.49                 | 8.09                
0.50       | 12.30           | 6.24                 | 6.07                


In [3]:
import sympy as sp
import numpy as np

# ==========================================
# PART 1: Symbolic Verification (SymPy)
# ==========================================
def verify_intensity_density():
    """
    Symbolically calculates the mixed partial derivative to verify Lemma 10.
    mu(b, d) = - d^2/(db dd) q_{b,d}
    """
    b, d, m = sp.symbols('b d m', real=True)
    
    # Delta is the length of the interval
    delta = d - b
    
    # Define q_{b,d} for b >= 0
    q_positive = 1 / (sp.exp(2 * m * delta) - 1)
    
    # Define q_{b,d} for b < 0
    q_negative = sp.exp(2 * m * b) / (sp.exp(2 * m * delta) - 1)
    
    # Calculate mixed partial derivatives: - d/db (d/dd (q))
    # For b >= 0
    dq_dd_pos = sp.diff(q_positive, d)
    mu_pos_raw = -sp.diff(dq_dd_pos, b)
    mu_pos_simplified = sp.simplify(mu_pos_raw)
    
    # For b < 0
    dq_dd_neg = sp.diff(q_negative, d)
    mu_neg_raw = -sp.diff(dq_dd_neg, b)
    mu_neg_simplified = sp.simplify(mu_neg_raw)
    
    print("--- Symbolic Verification Results ---")
    print(f"Density for b >= 0:\n{mu_pos_simplified}\n")
    print(f"Density for b < 0:\n{mu_neg_simplified}\n")


# ==========================================
# PART 2: Numerical Implementation (NumPy)
# ==========================================
def calculate_ph0_density(b, d, m):
    """
    Calculates the intensity measure density of PH_0 for Brownian motion with drift.
    Supports scalar values and NumPy arrays.
    
    Args:
        b: Birth time(s) / local minimum level (array-like or scalar)
        d: Death time(s) / local maximum level (array-like or scalar)
        m: Constant drift parameter (scalar)
        
    Returns:
        Density value(s) matching the input shape.
    """
    b = np.asarray(b)
    d = np.asarray(d)
    delta = d - b
    
    # Avoid division by zero on the diagonal where d = b
    # In persistence diagrams, density explodes near the diagonal (Delta -> 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        exp_2m_delta = np.exp(2 * m * delta)
        exp_2m_b = np.exp(2 * m * b)
        
        # Formula for b >= 0
        mu_pos = (4 * (m**2) * exp_2m_delta * (1 + exp_2m_delta)) / ((exp_2m_delta - 1)**3)
        
        # Formula for b < 0
        mu_neg = (8 * (m**2) * np.exp(4 * m * delta) * exp_2m_b) / ((exp_2m_delta - 1)**3)
        
        # Combine using np.where based on the condition b >= 0
        density = np.where(b >= 0, mu_pos, mu_neg)
        
    return density

# --- Example Usage ---
if __name__ == "__main__":
    verify_intensity_density()
    
    # Test numerical evaluation
    test_b = 1.0
    test_d = 2.0
    test_m = 0.5
    
    val = calculate_ph0_density(test_b, test_d, test_m)
    print(f"Numerical evaluation for b={test_b}, d={test_d}, m={test_m}: {val:.4f}")

--- Symbolic Verification Results ---
Density for b >= 0:
4*m**2*(exp(2*m*(b - d)) + 1)*exp(2*m*(b - d))/(1 - exp(2*m*(b - d)))**3

Density for b < 0:
8*m**2*exp(2*m*(2*b - d))/(1 - exp(2*m*(b - d)))**3

Numerical evaluation for b=1.0, d=2.0, m=0.5: 1.9923
